In [1]:
# Install timm for models and ptflops for efficiency metrics
!pip install -q timm ptflops gdown

In [2]:
import torch
import timm
import os
from ptflops import get_model_complexity_info

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Quick test of your first model selection
model = timm.create_model('resnet50', pretrained=True)
print("ResNet50 loaded successfully!")

# Directory to save your plots, tables, and model checkpoints
os.makedirs('/kaggle/working/results/plots', exist_ok=True)
os.makedirs('/kaggle/working/results/checkpoints', exist_ok=True)

PyTorch version: 2.9.0+cu126
GPU Available: True
GPU Model: Tesla T4


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ResNet50 loaded successfully!


In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# 1. Define standard transforms (ImageNet normalization is standard for timm models)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the full dataset (Assuming you downloaded/unzipped it to /kaggle/working/data/AID)
# Replace with your actual path
data_dir = '/kaggle/input/datasets/adarshguduru/gnr638-ass2/train_data' 
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# 3. Stratified Split (80% Train, 20% Val)
# Note: For the assignment, use a fixed seed for reproducibility!
seed = 42 
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], 
                                          generator=torch.Generator().manual_seed(seed))

# 4. DataLoaders (Optimized for 2 GPUs)
# Since you have 2 GPUs, you can double your batch size safely.
batch_size = 64 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Classes: {len(full_dataset.classes)} | Total Images: {len(full_dataset)}")

Classes: 30 | Total Images: 6993


In [4]:
import timm
import torch.nn as nn

def get_model(model_name, num_classes=30):
    # Load pre-trained model
    model = timm.create_model(model_name, pretrained=True)
    
    # Identify and replace the head for each architecture
    if 'resnet' in model_name:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif 'efficientnet' in model_name:
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif 'convnext' in model_name:
        model.head.fc = nn.Linear(model.head.fc.in_features, num_classes)
    
    # Move to GPU and wrap for 2x GPU usage
    model = model.cuda()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    return model

# Load your big three
model_resnet = get_model('resnet50')
model_efficient = get_model('efficientnet_b0')
model_convnext = get_model('convnext_tiny')

print("Models prepared for 30-class classification on 2x T4 GPUs.")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Models prepared for 30-class classification on 2x T4 GPUs.


In [5]:
from ptflops import get_model_complexity_info

# Helper to report metrics
def report_efficiency(model, name):
    # Use the .module if it's wrapped in DataParallel
    target_model = model.module if isinstance(model, nn.DataParallel) else model
    
    macs, params = get_model_complexity_info(target_model, (3, 224, 224), 
                                            as_strings=True, 
                                            print_per_layer_stat=False)
    print(f"[{name}] MACs: {macs} | Params: {params}")

report_efficiency(model_resnet, "ResNet50")
report_efficiency(model_efficient, "EfficientNet-B0")
report_efficiency(model_convnext, "ConvNeXt-Tiny")

[ResNet50] MACs: 4.13 GMac | Params: 23.57 M
[EfficientNet-B0] MACs: 390.87 MMac | Params: 4.05 M
[ConvNeXt-Tiny] MACs: 4.48 GMac | Params: 27.84 M


#scenario 4.1

**freesing the backbone**

In [6]:
def freeze_backbone(model):
    # If using DataParallel, we need to access .module
    target_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    
    # 1. Freeze everything
    for param in target_model.parameters():
        param.requires_grad = False
    
    # 2. Unfreeze only the head (the last layer)
    # Different models have different names for the head in timm
    if hasattr(target_model, 'fc'): # ResNet
        for param in target_model.fc.parameters():
            param.requires_grad = True
    elif hasattr(target_model, 'classifier'): # EfficientNet
        for param in target_model.classifier.parameters():
            param.requires_grad = True
    elif hasattr(target_model, 'head'): # ConvNeXt
        for param in target_model.head.parameters():
            param.requires_grad = True
            
    return model

model_resnet_4_1 = freeze_backbone(model_resnet)
model_efficient_4_1 = freeze_backbone(model_efficient)
model_convnext_4_1 = freeze_backbone(model_convnext)

print("Backbones frozen. Only the classification heads are trainable.")

Backbones frozen. Only the classification heads are trainable.


# train function

In [7]:
import time

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=30):
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training Phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc = 100. * correct / total
        train_loss = running_loss / total

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.cuda(), labels.cuda()
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / val_total
        
        # Save History
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        duration = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Time: {duration:.1f}s")
        
    return history

# for resnet50

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model_resnet.parameters()), lr=1e-3)

print("Starting Linear Probe for ResNet50...")
resnet_history = train_model(model_resnet_4_1, train_loader, val_loader, criterion, optimizer, num_epochs=30)

Starting Linear Probe for ResNet50...
Epoch 1/30 | Train Acc: 40.60% | Val Acc: 65.12% | Time: 38.2s
Epoch 2/30 | Train Acc: 70.79% | Val Acc: 70.41% | Time: 31.0s
Epoch 3/30 | Train Acc: 76.58% | Val Acc: 73.27% | Time: 31.1s
Epoch 4/30 | Train Acc: 78.35% | Val Acc: 75.63% | Time: 30.9s
Epoch 5/30 | Train Acc: 80.98% | Val Acc: 78.63% | Time: 31.2s
Epoch 6/30 | Train Acc: 82.36% | Val Acc: 79.06% | Time: 31.1s
Epoch 7/30 | Train Acc: 83.96% | Val Acc: 79.99% | Time: 31.6s
Epoch 8/30 | Train Acc: 85.04% | Val Acc: 80.99% | Time: 31.2s
Epoch 9/30 | Train Acc: 85.75% | Val Acc: 82.06% | Time: 31.2s
Epoch 10/30 | Train Acc: 86.29% | Val Acc: 81.70% | Time: 31.2s
Epoch 11/30 | Train Acc: 86.93% | Val Acc: 82.84% | Time: 31.5s
Epoch 12/30 | Train Acc: 87.63% | Val Acc: 82.06% | Time: 31.4s
Epoch 13/30 | Train Acc: 88.40% | Val Acc: 82.63% | Time: 31.8s
Epoch 14/30 | Train Acc: 89.10% | Val Acc: 82.27% | Time: 31.6s
Epoch 15/30 | Train Acc: 89.65% | Val Acc: 82.99% | Time: 31.3s
Epoch 16/30

# visuvilization

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def plot_tsne(model, dataloader,model_name, device='cuda', num_samples=600):
    model.eval()
    features = []
    labels_list = []
    
    # Get the raw model object (unwrapped from DataParallel)
    target_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    
    with torch.no_grad():
        for i, (imgs, lbls) in enumerate(dataloader):
            imgs = imgs.to(device)
            
            # Use forward_features to get the output from the last feature block
            # This works for ResNet, EfficientNet, and ConvNeXt
            feat = target_model.forward_features(imgs)
            
            # If the feature map is 4D (Batch, Channels, Height, Width), 
            # we need to average pool it manually if it isn't already 2D.
            if len(feat.shape) == 4:
                feat = feat.mean([2, 3]) # Global Average Pooling
            
            features.append(feat.cpu().numpy())
            labels_list.append(lbls.numpy())
            
            if len(np.concatenate(features)) >= num_samples:
                break

    features = np.concatenate(features)[:num_samples]
    labels_list = np.concatenate(labels_list)[:num_samples]
    
    # Run t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    embeds = tsne.fit_transform(features)
    
    # Plotting
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeds[:, 0], embeds[:, 1], c=labels_list, cmap='tab20', alpha=0.7)
    plt.colorbar(scatter)
    plt.title(f"t-SNE Visualization: {model_name}")
    
    # Save the plot
    save_path = f'/kaggle/working/results/plots/{model_name}_tsne.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"t-SNE plot saved to {save_path}")
    plt.close() # Close to free up memory


# Run it


In [10]:
plot_tsne(model_resnet_4_1, val_loader, "ResNet50_4_1")

t-SNE plot saved to /kaggle/working/results/plots/ResNet50_4_1_tsne.png


In [11]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(model, dataloader, class_names, model_name, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm_perc, annot=False, cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Normalized Confusion Matrix: {model_name}')
    plt.xticks(rotation=90)
    
    # Save the plot
    save_path = f'/kaggle/working/results/plots/{model_name}_cm.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Confusion Matrix saved to {save_path}")
    plt.close()

# Get the class names from the original dataset
class_names = full_dataset.classes



In [12]:
# Run the plot
plot_confusion_matrix(model_resnet_4_1, val_loader, class_names,"ResNet50_4_1")

Confusion Matrix saved to /kaggle/working/results/plots/ResNet50_4_1_cm.png


In [13]:
optimizer_eff = torch.optim.Adam(filter(lambda p: p.requires_grad, model_efficient_4_1.parameters()), lr=1e-3)
print("\n--- Starting Linear Probe: EfficientNet-B0 ---")
eff_history = train_model(model_efficient_4_1, train_loader, val_loader, criterion, optimizer_eff, num_epochs=30)


--- Starting Linear Probe: EfficientNet-B0 ---
Epoch 1/30 | Train Acc: 57.51% | Val Acc: 74.27% | Time: 32.7s
Epoch 2/30 | Train Acc: 79.46% | Val Acc: 79.41% | Time: 31.7s
Epoch 3/30 | Train Acc: 83.39% | Val Acc: 82.06% | Time: 31.8s
Epoch 4/30 | Train Acc: 85.59% | Val Acc: 82.70% | Time: 31.8s
Epoch 5/30 | Train Acc: 87.58% | Val Acc: 82.84% | Time: 31.7s
Epoch 6/30 | Train Acc: 88.34% | Val Acc: 82.92% | Time: 31.9s
Epoch 7/30 | Train Acc: 90.04% | Val Acc: 84.92% | Time: 31.9s
Epoch 8/30 | Train Acc: 90.56% | Val Acc: 84.63% | Time: 31.8s
Epoch 9/30 | Train Acc: 91.15% | Val Acc: 84.35% | Time: 31.7s
Epoch 10/30 | Train Acc: 92.40% | Val Acc: 85.13% | Time: 31.8s
Epoch 11/30 | Train Acc: 92.78% | Val Acc: 84.13% | Time: 31.9s
Epoch 12/30 | Train Acc: 93.05% | Val Acc: 83.85% | Time: 31.9s
Epoch 13/30 | Train Acc: 94.07% | Val Acc: 85.35% | Time: 31.7s
Epoch 14/30 | Train Acc: 94.35% | Val Acc: 85.06% | Time: 31.7s
Epoch 15/30 | Train Acc: 94.98% | Val Acc: 84.77% | Time: 31.7s
E

In [14]:
plot_tsne(model_efficient_4_1, val_loader,"Efficientnet_b0_4_1")

t-SNE plot saved to /kaggle/working/results/plots/Efficientnet_b0_4_1_tsne.png


In [15]:
plot_confusion_matrix(model_efficient_4_1, val_loader, class_names,"Efficientnet_b0_4_1")

Confusion Matrix saved to /kaggle/working/results/plots/Efficientnet_b0_4_1_cm.png


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer_cnxt = torch.optim.Adam(filter(lambda p: p.requires_grad, model_convnext_4_1.parameters()), lr=1e-3)
print("\n--- Starting Linear Probe: ConvNeXt-Tiny ---")
cnxt_history = train_model(model_convnext_4_1, train_loader, val_loader, criterion, optimizer_cnxt, num_epochs=30)


--- Starting Linear Probe: ConvNeXt-Tiny ---
Epoch 1/30 | Train Acc: 73.76% | Val Acc: 87.56% | Time: 247.1s
Epoch 2/30 | Train Acc: 91.60% | Val Acc: 89.14% | Time: 247.4s
Epoch 3/30 | Train Acc: 94.44% | Val Acc: 90.28% | Time: 246.3s
Epoch 4/30 | Train Acc: 96.55% | Val Acc: 91.07% | Time: 247.1s
Epoch 5/30 | Train Acc: 97.46% | Val Acc: 90.78% | Time: 243.5s
Epoch 6/30 | Train Acc: 98.05% | Val Acc: 90.99% | Time: 244.6s
Epoch 7/30 | Train Acc: 98.50% | Val Acc: 91.28% | Time: 246.2s
Epoch 8/30 | Train Acc: 99.00% | Val Acc: 91.35% | Time: 247.6s
Epoch 9/30 | Train Acc: 99.21% | Val Acc: 91.14% | Time: 249.0s
Epoch 10/30 | Train Acc: 99.36% | Val Acc: 91.71% | Time: 245.3s
Epoch 11/30 | Train Acc: 99.59% | Val Acc: 91.07% | Time: 244.1s
Epoch 12/30 | Train Acc: 99.68% | Val Acc: 91.71% | Time: 244.6s
Epoch 13/30 | Train Acc: 99.82% | Val Acc: 91.35% | Time: 244.0s
Epoch 14/30 | Train Acc: 99.84% | Val Acc: 90.92% | Time: 243.1s
Epoch 15/30 | Train Acc: 99.84% | Val Acc: 91.64% | T

In [17]:
plot_tsne(model_convnext_4_1, val_loader,"ConvNeXt_4_1")

t-SNE plot saved to /kaggle/working/results/plots/ConvNeXt_4_1_tsne.png


In [18]:
plot_confusion_matrix(model_convnext_4_1, val_loader, class_names, "ConvNeXt_4_1")

Confusion Matrix saved to /kaggle/working/results/plots/ConvNeXt_4_1_cm.png


In [19]:
import torch
import os

def save_checkpoint(model, filename="model_checkpoint.pth"):
    # Always save the model's state_dict
    target_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    
    # Define path in the working directory
    save_path = os.path.join('/kaggle/working/', filename)
    
    # Save the weights
    torch.save(target_model.state_dict(), save_path)
    print(f"Model weights saved to: {save_path}")

# Example Usage:
# save_checkpoint(model_convnext_4_1, "convnext_tiny_4_1.pth")

In [20]:
# Save your final models at the very end of your training script
save_checkpoint(model_resnet_4_1, "resnet_4_1.pth")
save_checkpoint(model_efficient_4_1, "efficient_4_1.pth")
save_checkpoint(model_convnext_4_1, "convnext_4_1.pth")

# Also save your training histories so you don't have to re-train to plot graphs
import json
with open('/kaggle/working/resnet_history.json', 'w') as f: json.dump(resnet_history, f)
with open('/kaggle/working/efficient_history.json', 'w') as f: json.dump(eff_history, f)
with open('/kaggle/working/convnext_history.json', 'w') as f: json.dump(cnxt_history, f)

Model weights saved to: /kaggle/working/resnet_4_1.pth
Model weights saved to: /kaggle/working/efficient_4_1.pth
Model weights saved to: /kaggle/working/convnext_4_1.pth
